# load data

In [23]:
import pandas as pd

gini_data = pd.read_csv('gini_data.csv')

print(gini_data.head())

                 GEO_ID                                               NAME  \
0  1400000US12107980000          Census Tract 9800, Putnam County, Florida   
1  1400000US37051980200  Census Tract 9802, Cumberland County, North Ca...   
2  1400000US12101990000           Census Tract 9900, Pasco County, Florida   
3  1400000US06073009902   Census Tract 99.02, San Diego County, California   
4  1400000US08014980100     Census Tract 9801, Broomfield County, Colorado   

  B19083_001E B19083_001M  
0          -0          **  
1          -0          **  
2          -0          **  
3          -0          **  
4          -0          **  


# calculate the mean, median and mode using the pandas library

In [24]:
gini_data['B19083_001E'] = pd.to_numeric(gini_data['B19083_001E'], errors='coerce')
                                                
mean_gini = gini_data['B19083_001E'].mean()
median_gini = gini_data['B19083_001E'].median()
mode_gini = gini_data['B19083_001E'].mode()[0]

print(f"Mean Gini Index: {mean_gini}")
print(f"Median Gini Index: {median_gini}")
print(f"Mode Gini Index: {mode_gini}")

Mean Gini Index: 0.42093657422424047
Median Gini Index: 0.4191
Mode Gini Index: -0.0


# calculate the mean,median and mode using the standard python library

In [25]:
from statistics import mean, median, mode

gini_values = [value for value in gini_data['B19083_001E'] if pd.notnull(value)]

mean_gini_std = sum(gini_values) / len(gini_values)
gini_values.sort()
if len(gini_values) % 2 == 1:
    median_gini_std = gini_values[len(gini_values) // 2]
else:
    median_gini_std = (gini_values[len(gini_values) // 2 - 1] + gini_values[len(gini_values) // 2]) / 2
mode_gini_std = max(set(gini_values), key=gini_values.count)

print(f"Mean Gini Index (Standard Library): {mean_gini_std}")
print(f"Median Gini Index (Standard Library): {median_gini_std}")
print(f"Mode Gini Index (Standard Library): {mode_gini_std}")

Mean Gini Index (Standard Library): 0.4209365742242404
Median Gini Index (Standard Library): 0.4191
Mode Gini Index (Standard Library): -0.0


# create the data vis using standard python library

In [26]:
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

df = gini_data
s = df["B19083_001E"].astype(str).str.strip().str.replace("−","-", regex=False)
vals = pd.to_numeric(s, errors="coerce").dropna().tolist()

W, H, P = 800, 300, 30
mn, mx = min(vals), max(vals)
rng = (mx - mn) or 1.0

xs, ys = [], []
for i, v in enumerate(vals):
    x = P + (W - 2*P) * (i / (len(vals) - 1 or 1))
    y = P + (H - 2*P) * ((mx - v) / rng)
    xs.append(x); ys.append(y)

svg = ET.Element("svg", xmlns="http://www.w3.org/2000/svg",
                 width=str(W), height=str(H))
ET.SubElement(svg, "rect", x="0", y="0", width=str(W), height=str(H), fill="white")
ET.SubElement(svg, "rect", x=str(P), y=str(P), width=str(W-2*P), height=str(H-2*P),
              fill="none", stroke="#ddd")

points = " ".join(f"{xs[i]},{ys[i]}" for i in range(len(vals)))
ET.SubElement(svg, "polyline", points=points, fill="none",
              stroke="black", **{"stroke-width": "1.5"})

title = ET.SubElement(svg, "text", x=str(W/2), y="20", fill="black",
                      **{"text-anchor": "middle", "font-size": "14"})
title.text = "B19083_001E (Gini Index) — Simple Line"

out = Path("gini_simple.svg")
ET.ElementTree(svg).write(out, encoding="unicode")
print("Saved:", out.resolve())


Saved: /Users/zcr/Desktop/gini_simple.svg
